In [1]:
import pandas as pd

df = pd.read_csv('./BunkerChurners_PearsonCleaned.csv')
print(df.head())

      Attrition_Flag  Dependent_count Education_Level Marital_Status  \
0  Existing Customer                3     High School        Married   
1  Existing Customer                5        Graduate         Single   
2  Existing Customer                3        Graduate        Married   
3  Existing Customer                4     High School        Unknown   
4  Existing Customer                3      Uneducated        Married   

  Income_Category Card_Category  Months_on_book  Total_Relationship_Count  \
0     $60K - $80K          Blue              39                         5   
1  Less than $40K          Blue              44                         6   
2    $80K - $120K          Blue              36                         4   
3  Less than $40K          Blue              34                         3   
4     $60K - $80K          Blue              21                         5   

   Months_Inactive_12_mon  Contacts_Count_12_mon  Credit_Limit  \
0                       1             

In [2]:
categorical_col = df.select_dtypes(include=['object', 'category']).columns.tolist()
print("Variabili categoriche:".upper())
for c in categorical_col:
    print("-", c)

VARIABILI CATEGORICHE:
- Attrition_Flag
- Education_Level
- Marital_Status
- Income_Category
- Card_Category


In [3]:
numeric_col = df.select_dtypes(include=['number']).columns.tolist()
print("variabili numeriche:".upper())
for i in numeric_col:
    print("-", i)

VARIABILI NUMERICHE:
- Dependent_count
- Months_on_book
- Total_Relationship_Count
- Months_Inactive_12_mon
- Contacts_Count_12_mon
- Credit_Limit
- Total_Revolving_Bal
- Total_Amt_Chng_Q4_Q1
- Total_Trans_Ct
- Total_Ct_Chng_Q4_Q1
- Avg_Utilization_Ratio


In [4]:
int_col = ['Dependent_count', 'Total_Relationship_Count', 'Months_Inactive_12_mon', 'Contacts_Count_12_mon']
float_col = [item for item in numeric_col if item not in int_col]
float_col

['Months_on_book',
 'Credit_Limit',
 'Total_Revolving_Bal',
 'Total_Amt_Chng_Q4_Q1',
 'Total_Trans_Ct',
 'Total_Ct_Chng_Q4_Q1',
 'Avg_Utilization_Ratio']

In [5]:
int_col

['Dependent_count',
 'Total_Relationship_Count',
 'Months_Inactive_12_mon',
 'Contacts_Count_12_mon']

Procediamo con la rimozione degli outlier

In [6]:
import numpy as np

print(f"Dimensione dataset originale: {df.shape}")
print("\nAnalisi outlier per variabile:")
print("-" * 60)

# Rimozione outlier con metodo IQR
df_clean = df.copy()
outlier_mask = pd.Series([False] * len(df_clean))

for col in float_col:
    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    col_outliers = (df_clean[col] < lower_bound) | (df_clean[col] > upper_bound)
    outlier_mask = outlier_mask | col_outliers
    
    n_outliers = col_outliers.sum()
    print(f"{col}: {n_outliers} outlier ({n_outliers/len(df_clean)*100:.2f}%)")

# Applicazione maschera
df_clean = df_clean[~outlier_mask]

print("-" * 60)
print(f"\nDimensione dataset pulito: {df_clean.shape}")
print(f"Righe rimosse: {len(df) - len(df_clean)} ({(len(df) - len(df_clean))/len(df)*100:.2f}%)")

print("\nDistribuzione Attrition_Flag dopo rimozione:")
print(df_clean['Attrition_Flag'].value_counts())

# Aggiorna il dataframe
df = df_clean

Dimensione dataset originale: (10127, 16)

Analisi outlier per variabile:
------------------------------------------------------------
Months_on_book: 386 outlier (3.81%)
Credit_Limit: 984 outlier (9.72%)
Total_Revolving_Bal: 0 outlier (0.00%)
Total_Amt_Chng_Q4_Q1: 396 outlier (3.91%)
Total_Trans_Ct: 2 outlier (0.02%)
Total_Ct_Chng_Q4_Q1: 394 outlier (3.89%)
Avg_Utilization_Ratio: 0 outlier (0.00%)
------------------------------------------------------------

Dimensione dataset pulito: (8183, 16)
Righe rimosse: 1944 (19.20%)

Distribuzione Attrition_Flag dopo rimozione:
Attrition_Flag
Existing Customer    6857
Attrited Customer    1326
Name: count, dtype: int64


In [7]:
# Salva il dataset dopo la rimozione degli outlier
df.to_csv('BunkerChurners_PearsonCleaned_OutliersRemoved.csv', index=False)

# Preprocessing variabili categoriche

- Attrition_Flag (target): Label Encoding (0/1) - è binaria
- Card_Category: One-Hot Encoding - ha poche categorie ordinate (Blue, Silver, Gold, Platinum)
- Education_Level: Ordinal Encoding - ha un ordine naturale (es. High School < Graduate < Post-Graduate)
- Marital_Status: One-Hot Encoding - nominale senza ordine
- Income_Category: Ordinal Encoding - ha ordine naturale (es. <40K < 40K-60K < 60K-80K, ecc.)

In [7]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder, LabelEncoder
from sklearn.model_selection import train_test_split

# ==========================================
# PREPARAZIONE DATI
# ==========================================

# Definizione colonne per tipo
variabili_numeriche = [
    'Dependent_count', 'Months_on_book', 'Total_Relationship_Count',
    'Months_Inactive_12_mon', 'Contacts_Count_12_mon', 'Credit_Limit',
    'Total_Revolving_Bal', 'Total_Amt_Chng_Q4_Q1', 'Total_Trans_Ct',
    'Total_Ct_Chng_Q4_Q1', 'Avg_Utilization_Ratio'
]

# Categoriche ordinali con ordine specifico
education_order = [
    ['Unknown', 'Uneducated', 'High School', 'College', 
     'Graduate', 'Post-Graduate', 'Doctorate']
]

income_order = [
    ['Unknown', 'Less than $40K', '$40K - $60K', '$60K - $80K', 
     '$80K - $120K', '$120K +']
]

# Categoriche nominali (per One-Hot)
categoriche_nominali = ['Card_Category', 'Marital_Status']

# Categoriche ordinali (colonne)
categoriche_ordinali_edu = ['Education_Level']
categoriche_ordinali_inc = ['Income_Category']

# ==========================================
# ENCODING DEL TARGET
# ==========================================

# Separa X e y
X = df.drop(columns=['Attrition_Flag'])
y = df['Attrition_Flag']

# Encoding target
le = LabelEncoder()
y_encoded = le.fit_transform(y)

print(f"Target encoding: {dict(zip(le.classes_, le.transform(le.classes_)))}")
print(f"Distribuzione target: {pd.Series(y_encoded).value_counts().to_dict()}")

# ==========================================
# SPLIT TRAIN-TEST
# ==========================================
# IMPORTANTE: Split PRIMA del preprocessing per evitare data leakage

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f"\nTrain set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

# ==========================================
# CREAZIONE PIPELINE
# ==========================================

# Transformer per variabili numeriche (standardizzazione)
numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

# Transformer per categoriche ordinali - Education
ordinal_edu_transformer = Pipeline(steps=[
    ('ordinal', OrdinalEncoder(categories=education_order, handle_unknown='use_encoded_value', unknown_value=-1))
])

# Transformer per categoriche ordinali - Income
ordinal_inc_transformer = Pipeline(steps=[
    ('ordinal', OrdinalEncoder(categories=income_order, handle_unknown='use_encoded_value', unknown_value=-1))
])

# Transformer per categoriche nominali (One-Hot)
categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False))
])

# Combina tutti i transformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, variabili_numeriche),
        ('ord_edu', ordinal_edu_transformer, categoriche_ordinali_edu),
        ('ord_inc', ordinal_inc_transformer, categoriche_ordinali_inc),
        ('cat', categorical_transformer, categoriche_nominali)
    ],
    remainder='drop'  # Elimina colonne non specificate
)

Target encoding: {'Attrited Customer': 0, 'Existing Customer': 1}
Distribuzione target: {1: 6857, 0: 1326}

Train set: (6546, 15)
Test set: (1637, 15)


In [8]:
# ==========================================
# APPLICAZIONE PREPROCESSING
# ==========================================

# Fit sul train, transform su train e test
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print(f"\nX_train processato: {X_train_processed.shape}")
print(f"X_test processato: {X_test_processed.shape}")


X_train processato: (6546, 19)
X_test processato: (1637, 19)


In [9]:
# ==========================================
# NOMI DELLE FEATURE (opzionale ma utile)
# ==========================================

# Estrai i nomi delle feature dopo il preprocessing
feature_names = []

# Numeriche
feature_names.extend(variabili_numeriche)

# Ordinali Education
feature_names.extend(categoriche_ordinali_edu)

# Ordinali Income
feature_names.extend(categoriche_ordinali_inc)

# One-Hot (ottieni i nomi dalle categorie)
if hasattr(preprocessor.named_transformers_['cat']['onehot'], 'get_feature_names_out'):
    onehot_features = preprocessor.named_transformers_['cat']['onehot'].get_feature_names_out(categoriche_nominali)
    feature_names.extend(onehot_features)

print(f"\nFeature finali ({len(feature_names)}):")
for i, name in enumerate(feature_names):
    print(f"  {i}: {name}")


Feature finali (19):
  0: Dependent_count
  1: Months_on_book
  2: Total_Relationship_Count
  3: Months_Inactive_12_mon
  4: Contacts_Count_12_mon
  5: Credit_Limit
  6: Total_Revolving_Bal
  7: Total_Amt_Chng_Q4_Q1
  8: Total_Trans_Ct
  9: Total_Ct_Chng_Q4_Q1
  10: Avg_Utilization_Ratio
  11: Education_Level
  12: Income_Category
  13: Card_Category_Gold
  14: Card_Category_Platinum
  15: Card_Category_Silver
  16: Marital_Status_Married
  17: Marital_Status_Single
  18: Marital_Status_Unknown


In [11]:
# ==========================================
# ESEMPIO: PIPELINE COMPLETA CON MODELLO
# ==========================================

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, precision_recall_curve, auc
from sklearn.utils.class_weight import compute_class_weight

# Calcolo class weights
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weight_dict = {i: weight for i, weight in enumerate(class_weights)}

print("\n" + "=" * 60)
print("ANALISI SBILANCIAMENTO CLASSI")
print("=" * 60)
print(f"\nDistribuzione train set:")
unique, counts = np.unique(y_train, return_counts=True)
for cls, cnt in zip(unique, counts):
    print(f"  Classe {cls} ({le.classes_[cls]}): {cnt} ({cnt/len(y_train)*100:.2f}%)")

print(f"\nClass weights calcolati: {class_weight_dict}")
print(f"  Existing Customer (0): peso {class_weight_dict[0]:.3f}")
print(f"  Attrited Customer (1): peso {class_weight_dict[1]:.3f}")

# Pipeline completa: preprocessing + modello con class_weight
full_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        random_state=42, 
        n_estimators=400,
        class_weight=class_weight_dict,  # Applica i pesi
        max_depth=15,
        min_samples_split=10,
        min_samples_leaf=4
    ))
])

# Fit del modello
print("\n" + "=" * 60)
print("Training del modello con class weighting...")
print("=" * 60)
full_pipeline.fit(X_train, y_train)

# Predizioni
y_pred = full_pipeline.predict(X_test)
y_pred_proba = full_pipeline.predict_proba(X_test)[:, 1]


ANALISI SBILANCIAMENTO CLASSI

Distribuzione train set:
  Classe 0 (Attrited Customer): 1061 (16.21%)
  Classe 1 (Existing Customer): 5485 (83.79%)

Class weights calcolati: {0: 3.0848256361922712, 1: 0.596718322698268}
  Existing Customer (0): peso 3.085
  Attrited Customer (1): peso 0.597

Training del modello con class weighting...


In [12]:
# ==========================================
# VALUTAZIONE COMPLETA PER DATASET SBILANCIATO
# ==========================================

print("\nRisultati sul test set:")
print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
print(cm)
print(f"\nTrue Negatives: {cm[0,0]}")
print(f"False Positives: {cm[0,1]}")
print(f"False Negatives: {cm[1,0]}")
print(f"True Positives: {cm[1,1]}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

# Metriche aggiuntive importanti per classi sbilanciate
from sklearn.metrics import balanced_accuracy_score, f1_score, recall_score, precision_score

print("=" * 60)
print("METRICHE CHIAVE PER CLASSE MINORITARIA (Attrited Customer)")
print("=" * 60)
print(f"Precision (classe 1): {precision_score(y_test, y_pred):.3f}")
print(f"Recall (classe 1): {recall_score(y_test, y_pred):.3f}")
print(f"F1-Score (classe 1): {f1_score(y_test, y_pred):.3f}")
print(f"Balanced Accuracy: {balanced_accuracy_score(y_test, y_pred):.3f}")
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_pred_proba):.3f}")

# Precision-Recall AUC (migliore per dataset sbilanciati)
precision_vals, recall_vals, _ = precision_recall_curve(y_test, y_pred_proba)
pr_auc = auc(recall_vals, precision_vals)
print(f"PR-AUC Score: {pr_auc:.3f}")



Risultati sul test set:

Confusion Matrix:
[[ 198   67]
 [  78 1294]]

True Negatives: 198
False Positives: 67
False Negatives: 78
True Positives: 1294

Classification Report:
                   precision    recall  f1-score   support

Attrited Customer       0.72      0.75      0.73       265
Existing Customer       0.95      0.94      0.95      1372

         accuracy                           0.91      1637
        macro avg       0.83      0.85      0.84      1637
     weighted avg       0.91      0.91      0.91      1637

METRICHE CHIAVE PER CLASSE MINORITARIA (Attrited Customer)
Precision (classe 1): 0.951
Recall (classe 1): 0.943
F1-Score (classe 1): 0.947
Balanced Accuracy: 0.845
ROC-AUC Score: 0.956
PR-AUC Score: 0.991


In [ ]:
# ==========================================
# SALVARE E RIUTILIZZARE LA PIPELINE
# ==========================================

# Per salvare la pipeline completa (preprocessing + modello):
# import joblib
# joblib.dump(full_pipeline, 'model_pipeline.pkl')

# Per caricarla e usarla su nuovi dati:
# loaded_pipeline = joblib.load('model_pipeline.pkl')
# predictions = loaded_pipeline.predict(new_data)